In [3]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
import urllib.request

url = "https://raw.githubusercontent.com/rickiepark/llm-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
file_path = "the_verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the_verdict.txt', <http.client.HTTPMessage at 0x153bd7fb250>)

In [5]:
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()

In [6]:
tatal_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print(f"total characters: {tatal_characters}")
print(f"total tokens: {total_tokens}")

total characters: 20479
total tokens: 5145


In [7]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
validate_data = text_data[split_idx:]

In [8]:
from previous_chapters import create_dataloader_v1, GPTModel

In [9]:
import torch
torch.manual_seed(123)

In [10]:
import os, sys
sys.path.append(os.pardir)
from chapter4.gpt_config import GPT_CONFIG_124M

In [11]:
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    validate_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [12]:
print("train_loader: ")
for x, y in train_loader:
    print("\t", x.shape, y.shape)

print("valid_loader: ")
for x, y in val_loader:
    print("\t",x.shape, y.shape)

train_loader: 
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
	 torch.Size([2, 256]) torch.Size([2, 256])
valid_loader: 
	 torch.Size([2, 256]) torch.Size([2, 256])


In [13]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

In [14]:
def clac_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) ==0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [15]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = clac_loss_loader(train_loader, model, device)
    validate_loss = clac_loss_loader(val_loader, model, device)

print("train_loss: ", train_loss)
print("validate_loss: ", validate_loss)

train_loss:  10.987583584255642
validate_loss:  10.981106758117676
